# Industrial Telemetry Analytics — Machine Failure Pattern Study
**Author:** Puru Pandey

Research Questions:
1. Which machine units / factories fail earliest?
2. Can custom feature engineering improve failure signal detection?
3. What sensors are most predictive?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.3f}'.format)
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette('husl')

print(f'numpy {np.__version__} | pandas {pd.__version__}')

## Phase 1: Data Ingestion

In [ ]:
COLUMNS = [
    'unit_id', 'cycle',
    'op_setting_1', 'op_setting_2', 'op_setting_3',
    *[f'sensor_{i:02d}' for i in range(1, 22)]
]

def generate_synthetic_telemetry(n_units=80, seed=42):
    """Generate synthetic telemetry data mirroring factory sensor streams."""
    np.random.seed(seed)
    records = []
    factory_groups = ['Factory_A', 'Factory_B', 'Factory_C', 'Factory_D']
    machine_types = [f'MachineType_{chr(65+i)}' for i in range(9)]
    
    for unit_id in range(1, n_units + 1):
        factory = factory_groups[unit_id % 4]
        machine_type = machine_types[unit_id % 9]
        total_cycles = np.random.randint(150, 350)
        
        for cycle in range(1, total_cycles + 1):
            degradation = cycle / total_cycles
            row = {
                'unit_id': unit_id, 'cycle': cycle,
                'factory': factory, 'machine_type': machine_type,
                'op_setting_1': np.random.choice([0.0, 0.25, 0.42]),
                'op_setting_2': np.random.choice([0.0, 0.0003, 14.62]),
                'op_setting_3': 100.0,
                'total_life': total_cycles,
                'rul': total_cycles - cycle,
            }
            for s in range(1, 22):
                base = np.random.uniform(200, 1500)
                noise = np.random.normal(0, base * 0.02)
                direction = 1 if s % 3 != 0 else -1
                drift = direction * degradation * base * np.random.uniform(0.05, 0.25)
                row[f'sensor_{s:02d}'] = round(base + drift + noise, 4)
            records.append(row)
    
    df = pd.DataFrame(records)
    print(f'generated {len(df):,} records | {n_units} units | 4 factories | 9 machine types')
    return df

import os
if os.path.exists('./data/train_FD001.txt'):
    df = pd.read_csv('./data/train_FD001.txt', sep='\s+', header=None, names=COLUMNS)
else:
    print('NASA data not found - using synthetic dataset')
    df = generate_synthetic_telemetry(n_units=80, seed=42)

print(f'shape: {df.shape}')
df.head()

## Phase 2: EDA

In [ ]:
# basic dataset overview
print(f'records: {len(df):,}')
print(f'units: {df["unit_id"].nunique()}')
print(f'missing: {df.isnull().sum().sum()}')
print(f'cycle range: {df["cycle"].min()} - {df["cycle"].max()}')
df.describe()

In [ ]:
# machine lifetimes
unit_lifetimes = df.groupby('unit_id')['cycle'].max().reset_index()
unit_lifetimes.columns = ['unit_id', 'total_cycles']

if 'factory' in df.columns:
    unit_meta = df.drop_duplicates('unit_id')[['unit_id', 'factory', 'machine_type']]
    unit_lifetimes = unit_lifetimes.merge(unit_meta, on='unit_id')

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(unit_lifetimes['total_cycles'], bins=25, color='#4C72B0', edgecolor='white', alpha=0.85)
axes[0].axvline(unit_lifetimes['total_cycles'].mean(), color='red', linestyle='--',
                label=f'Mean: {unit_lifetimes["total_cycles"].mean():.0f}')
axes[0].set_xlabel('Total Cycles Before Failure')
axes[0].set_ylabel('Units')
axes[0].set_title('Machine Lifetime Distribution')
axes[0].legend()

if 'factory' in unit_lifetimes.columns:
    factory_stats = unit_lifetimes.groupby('factory')['total_cycles'].mean().sort_values()
    colors = ['#d62728' if v == factory_stats.min() else '#2ca02c' for v in factory_stats.values]
    axes[1].barh(factory_stats.index, factory_stats.values, color=colors)
    axes[1].set_title('Avg Lifetime by Factory (red = highest failure rate)')
    axes[1].set_xlabel('Avg Cycles')
    
    worst = factory_stats.idxmin()
    print(f'>> {worst} has highest failure rate: avg {factory_stats.min():.0f} cycles')

plt.tight_layout()
plt.show()